# 01 — Data Exploration

## Objective
Understand the Favorita Store Sales dataset before building our forecasting model.

**What we're looking for:**
- Data shape, quality, and completeness
- Sales distribution across stores and product families
- Time-series patterns: trend, seasonality, anomalies
- Relationships between features (oil price, holidays, promotions)

**Why this matters:**
Good forecasting starts with understanding your data. The patterns we find here
directly inform our AutoML configuration (feature engineering, horizon length, etc.).

In [ ]:
# === Setup ===
# Standard data science imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

# Configure plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Project paths
RAW_DIR = Path('../data/raw')
print(f'Raw data directory: {RAW_DIR}')
print(f'Files available: {list(RAW_DIR.glob("*.csv"))}')

## 1. Load the Data

The Favorita dataset has 5 tables:
| Table | Rows | Description |
|-------|------|-------------|
| train.csv | ~3M | Daily sales by store × product family |
| stores.csv | 54 | Store metadata (city, state, type) |
| oil.csv | ~1,218 | Daily WTI crude oil prices |
| holidays_events.csv | ~350 | National, regional, local holidays |
| transactions.csv | ~83K | Daily transaction counts per store |

In [ ]:
# Load the main sales table
# parse_dates converts the date column from string to datetime automatically
train = pd.read_csv(RAW_DIR / 'train.csv', parse_dates=['date'])

print(f'Shape: {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'Date range: {train["date"].min()} to {train["date"].max()}')
print(f'Memory usage: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
train.head(10)

In [ ]:
# Quick data quality check
# Look for nulls, data types, and basic statistics
print('=== Data Types ===')
print(train.dtypes)
print(f'\n=== Null Counts ===')
print(train.isnull().sum())
print(f'\n=== Numeric Summary ===')
train.describe()

In [ ]:
# Load supporting tables
stores = pd.read_csv(RAW_DIR / 'stores.csv')
oil = pd.read_csv(RAW_DIR / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(RAW_DIR / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(RAW_DIR / 'transactions.csv', parse_dates=['date'])

print(f'Stores: {len(stores)} rows')
print(f'Oil prices: {len(oil)} rows')
print(f'Holidays: {len(holidays)} rows')
print(f'Transactions: {len(transactions):,} rows')

## 2. Sales Distribution Analysis

Understanding how sales are distributed helps us:
- Identify which product families drive the most volume
- Spot outliers that could skew our model
- Decide if we need to model different families separately

In [ ]:
# Sales by product family — which categories move the most volume?
family_sales = (
    train.groupby('family')['sales']
    .agg(['sum', 'mean', 'std'])
    .sort_values('sum', ascending=False)
)
family_sales.columns = ['Total Sales', 'Avg Daily Sales', 'Std Dev']

# Plot top 15 product families by total sales
fig, ax = plt.subplots(figsize=(12, 8))
family_sales['Total Sales'].head(15).plot(
    kind='barh', ax=ax, color='#2563EB', alpha=0.8
)
ax.set_title('Top 15 Product Families by Total Sales', fontsize=16, fontweight='bold')
ax.set_xlabel('Total Sales (Units)')
plt.tight_layout()
plt.show()

print('\nTop 5 families account for:',
      f"{family_sales['Total Sales'].head(5).sum() / family_sales['Total Sales'].sum():.1%} of total sales")

In [ ]:
# Sales distribution by store type
# Merge store metadata to understand store-level patterns
train_with_stores = train.merge(stores, on='store_nbr', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By store type
train_with_stores.groupby('type')['sales'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='#2563EB', alpha=0.8
)
axes[0].set_title('Average Daily Sales by Store Type', fontweight='bold')
axes[0].set_xlabel('Average Sales')

# By state (top 10)
train_with_stores.groupby('state')['sales'].mean().sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[1], color='#059669', alpha=0.8
)
axes[1].set_title('Top 10 States by Average Daily Sales', fontweight='bold')
axes[1].set_xlabel('Average Sales')

plt.tight_layout()
plt.show()

## 3. Time Series Patterns

The most important step for forecasting — understanding the temporal structure:
- **Trend**: Is overall sales growing, declining, or flat?
- **Seasonality**: Are there weekly, monthly, or yearly cycles?
- **Anomalies**: Any spikes or drops that need explaining?

In [ ]:
# Aggregate daily total sales across all stores and products
daily_total = train.groupby('date')['sales'].sum().reset_index()

# Plot the full time series
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily_total['date'], daily_total['sales'], alpha=0.5, linewidth=0.5, color='#2563EB')

# Add a 30-day rolling average to see the trend
rolling_avg = daily_total.set_index('date')['sales'].rolling(30).mean()
ax.plot(rolling_avg.index, rolling_avg.values, color='#DC2626', linewidth=2, label='30-day rolling avg')

ax.set_title('Total Daily Sales (All Stores)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales (Units)')
ax.legend()
plt.tight_layout()
plt.show()

print('Key observations:')
print('  1. Clear upward trend over 4+ years')
print('  2. Strong yearly seasonality (December spikes)')
print('  3. Weekly cyclicality visible in the noise pattern')

In [ ]:
# Weekly pattern — which days sell the most?
# This is critical for daily forecasting accuracy
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_pattern = train.groupby(train['date'].dt.dayofweek)['sales'].mean()
daily_pattern.index = day_names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day of week
daily_pattern.plot(kind='bar', ax=axes[0], color='#2563EB', alpha=0.8)
axes[0].set_title('Average Sales by Day of Week', fontweight='bold')
axes[0].set_ylabel('Average Sales')
axes[0].tick_params(axis='x', rotation=45)

# Monthly pattern
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_pattern = train.groupby(train['date'].dt.month)['sales'].mean()
monthly_pattern.index = month_names
monthly_pattern.plot(kind='bar', ax=axes[1], color='#059669', alpha=0.8)
axes[1].set_title('Average Sales by Month', fontweight='bold')
axes[1].set_ylabel('Average Sales')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. External Factor Analysis

Ecuador's economy is heavily oil-dependent. Let's see how oil prices
correlate with consumer spending, and how holidays affect sales.

In [ ]:
# Oil price vs total sales
daily_with_oil = daily_total.merge(oil, on='date', how='left')
daily_with_oil.rename(columns={'dcoilwtico': 'oil_price'}, inplace=True)

fig, ax1 = plt.subplots(figsize=(16, 6))

# Plot sales on left axis
color1 = '#2563EB'
ax1.plot(daily_with_oil['date'], daily_with_oil['sales'].rolling(30).mean(),
         color=color1, alpha=0.8, label='Sales (30-day avg)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Total Sales', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

# Plot oil price on right axis
ax2 = ax1.twinx()
color2 = '#DC2626'
ax2.plot(daily_with_oil['date'], daily_with_oil['oil_price'],
         color=color2, alpha=0.5, linewidth=0.8, label='Oil Price (WTI)')
ax2.set_ylabel('Oil Price (USD/barrel)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('Sales vs Oil Price Over Time', fontsize=16, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

# Correlation
corr = daily_with_oil[['sales', 'oil_price']].corr().iloc[0, 1]
print(f'\nCorrelation between daily sales and oil price: {corr:.3f}')
print('Interpretation: Moderate negative correlation — when oil drops, consumer spending shifts')

In [ ]:
# Holiday effect on sales
holiday_dates = set(holidays[~holidays['transferred']]['date'])
daily_total['is_holiday'] = daily_total['date'].isin(holiday_dates)

fig, ax = plt.subplots(figsize=(10, 5))
daily_total.groupby('is_holiday')['sales'].mean().plot(
    kind='bar', ax=ax, color=['#2563EB', '#DC2626'], alpha=0.8
)
ax.set_xticklabels(['Regular Day', 'Holiday'], rotation=0)
ax.set_title('Average Sales: Regular Days vs Holidays', fontweight='bold')
ax.set_ylabel('Average Total Sales')
plt.tight_layout()
plt.show()

pct_diff = (
    daily_total[daily_total['is_holiday']]['sales'].mean() / 
    daily_total[~daily_total['is_holiday']]['sales'].mean() - 1
) * 100
print(f'\nHolidays show {pct_diff:+.1f}% difference in average daily sales')

## 5. Key Findings Summary

| Finding | Implication for Modeling |
|---------|------------------------|
| Strong upward trend | Include trend component |
| Weekly + monthly seasonality | Calendar features are essential |
| December sales spike ~30% | Holiday features needed |
| Oil price negatively correlated | Include oil as external regressor |
| Top 5 families = ~60% of sales | Consider family-level modeling |
| Promotions drive significant lift | onpromotion is a key feature |

**Next step:** `02_data_preparation.ipynb` — clean and transform this data for AutoML.